In [9]:
! pip install transformers==4.35.2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [10]:
# translator_lib.py
# transformers==4.35.2 전제 (DynamicCache 사용 안 함)

from dataclasses import dataclass
from typing import Tuple, List, Literal

import torch
import torch.nn as nn
import torch.nn.functional as F

KVKind = Literal["k", "v"]


# ---------------------------
# KV pack/unpack helpers
# ---------------------------

@dataclass
class ModelKVSpec:
    n_layers: int
    n_heads: int
    head_dim: int
    hidden_size: int  # n_heads * head_dim


def pack_past_key_values(
    past_key_values: Tuple[Tuple[torch.Tensor, torch.Tensor], ...]
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    HF GPT2 past_key_values:
      tuple(n_layers) of (k, v)
      k,v: [B, n_heads, S, head_dim]
    return:
      K,V: [B, S, L, hidden_size]  where hidden_size=n_heads*head_dim
    """
    keys: List[torch.Tensor] = []
    values: List[torch.Tensor] = []

    for (k, v) in past_key_values:
        # [B, nH, S, Hd] -> [B, S, nH, Hd] -> [B, S, nH*Hd]
        k2 = k.permute(0, 2, 1, 3).contiguous().view(k.size(0), k.size(2), -1)
        v2 = v.permute(0, 2, 1, 3).contiguous().view(v.size(0), v.size(2), -1)
        keys.append(k2)
        values.append(v2)

    K = torch.stack(keys, dim=2)   # [B, S, L, D]
    V = torch.stack(values, dim=2) # [B, S, L, D]
    return K, V


def unpack_past_key_values(
    K: torch.Tensor,
    V: torch.Tensor,
    spec: ModelKVSpec
) -> Tuple[Tuple[torch.Tensor, torch.Tensor], ...]:
    """
    K,V: [B, S, L, hidden_size]
    return HF past_key_values tuple:
      k,v: [B, n_heads, S, head_dim]
    """
    B, S, L, D = K.shape
    assert L == spec.n_layers, (L, spec.n_layers)
    assert D == spec.hidden_size, (D, spec.hidden_size)

    out = []
    for layer in range(L):
        k = K[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        v = V[:, :, layer, :].contiguous().view(B, S, spec.n_heads, spec.head_dim).permute(0, 2, 1, 3).contiguous()
        out.append((k, v))
    return tuple(out)


def cosine_sim_flat(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> float:
    a1 = a.detach().float().reshape(-1)
    b1 = b.detach().float().reshape(-1)
    denom = (a1.norm() * b1.norm()).clamp_min(eps)
    return float(torch.dot(a1, b1) / denom)


# ---------------------------
# Cross-attention translator blocks (shared)
# ---------------------------

class CrossAttnBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, mlp_ratio: float = 4.0, dropout: float = 0.0):
        super().__init__()
        self.ln_q = nn.LayerNorm(d_model)
        self.ln_kv = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, int(d_model * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(d_model * mlp_ratio), d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mem: torch.Tensor) -> torch.Tensor:
        q = self.ln_q(x)
        kv = self.ln_kv(mem)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


# ---------------------------
# Adapters:
# - K/V share cross-attn blocks
# - ONLY projections differ for K vs V
# ---------------------------

class LocalToSharedAdapterKV(nn.Module):
    """
    T[mi -> Σ] for either K or V:
      input:  [B,S,L,Di]
      output: [B,S,Q]
    """
    def __init__(
        self,
        n_layers: int,
        d_in: int,
        q_dim: int,
        d_model: int = 256,
        n_heads: int = 8,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.n_layers = n_layers
        self.d_in = d_in
        self.q_dim = q_dim
        self.d_model = d_model

        self.in_ln = nn.LayerNorm(d_in)
        self.in_proj_k = nn.Linear(d_in, d_model)
        self.in_proj_v = nn.Linear(d_in, d_model)

        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        self.out_ln = nn.LayerNorm(n_layers * d_model)
        self.out_proj_k = nn.Linear(n_layers * d_model, q_dim)
        self.out_proj_v = nn.Linear(n_layers * d_model, q_dim)

        self.act = nn.GELU()

    def forward(self, x: torch.Tensor, kind: KVKind) -> torch.Tensor:
        B, S, L, Di = x.shape
        assert L == self.n_layers and Di == self.d_in

        x = self.in_ln(x)
        if kind == "k":
            h = self.act(self.in_proj_k(x))
        elif kind == "v":
            h = self.act(self.in_proj_v(x))
        else:
            raise ValueError(kind)

        cur = h[:, :, 0, :]
        outs = []
        for layer_idx, blk in enumerate(self.blocks):
            mem = h[:, :, layer_idx, :]
            cur = blk(cur, mem)
            outs.append(cur)

        cat = self.out_ln(torch.cat(outs, dim=-1))
        if kind == "k":
            y = self.act(self.out_proj_k(cat))
        else:
            y = self.act(self.out_proj_v(cat))
        return y  # [B,S,Q]


class SharedToLocalAdapterKV(nn.Module):
    """
    T[Σ -> mi] for either K or V:
      input:  [B,S,Q]
      output: [B,S,L,Di]
    """
    def __init__(
        self,
        n_layers: int,
        q_dim: int,
        d_out: int,
        d_model: int = 256,
        n_heads: int = 8,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
    ):
        super().__init__()
        assert q_dim % n_layers == 0, f"q_dim({q_dim}) must be divisible by n_layers({n_layers})"

        self.n_layers = n_layers
        self.q_dim = q_dim
        self.d_out = d_out
        self.d_model = d_model

        self.q_per_layer = q_dim // n_layers

        self.in_ln = nn.LayerNorm(self.q_per_layer)
        self.in_proj_k = nn.Linear(self.q_per_layer, d_model)
        self.in_proj_v = nn.Linear(self.q_per_layer, d_model)

        self.blocks = nn.ModuleList([CrossAttnBlock(d_model, n_heads, mlp_ratio, dropout) for _ in range(n_layers)])

        self.out_ln = nn.LayerNorm(n_layers * d_model)
        self.out_proj_k = nn.Linear(n_layers * d_model, n_layers * d_out)
        self.out_proj_v = nn.Linear(n_layers * d_model, n_layers * d_out)

        self.act = nn.GELU()

    def forward(self, x: torch.Tensor, kind: KVKind) -> torch.Tensor:
        B, S, Q = x.shape
        assert Q == self.q_dim

        x = x.view(B, S, self.n_layers, self.q_per_layer)
        x = self.in_ln(x)

        if kind == "k":
            h = self.act(self.in_proj_k(x))
        elif kind == "v":
            h = self.act(self.in_proj_v(x))
        else:
            raise ValueError(kind)

        cur = h[:, :, 0, :]
        outs = []
        for layer_idx, blk in enumerate(self.blocks):
            mem = h[:, :, layer_idx, :]
            cur = blk(cur, mem)
            outs.append(cur)

        cat = self.out_ln(torch.cat(outs, dim=-1))
        if kind == "k":
            y = self.act(self.out_proj_k(cat))
        else:
            y = self.act(self.out_proj_v(cat))

        y = y.view(B, S, self.n_layers, self.d_out)
        return y  # [B,S,L,d_out]


# ---------------------------
# One-way translator: B (gpt2-medium) -> A (gpt2)
# ---------------------------

class OneWayKVTranslator_B2A(nn.Module):
    """
    B -> Σ -> A
    학습/추론 모두 이 방향만 사용.
    """
    def __init__(
        self,
        b_layers: int, b_hidden: int,
        a_layers: int, a_hidden: int,
        q_dim: int = 1536,
        d_model: int = 256,
        n_heads: int = 8,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.B_to_S = LocalToSharedAdapterKV(b_layers, b_hidden, q_dim, d_model, n_heads, dropout=dropout)
        self.S_to_A = SharedToLocalAdapterKV(a_layers, q_dim, a_hidden, d_model, n_heads, dropout=dropout)

    @torch.no_grad()
    def translate(self, K_B: torch.Tensor, V_B: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        K_star = self.B_to_S(K_B, "k")
        V_star = self.B_to_S(V_B, "v")
        K_A = self.S_to_A(K_star, "k")
        V_A = self.S_to_A(V_star, "v")
        return K_A, V_A

    def loss(self, K_B, V_B, K_A_target, V_A_target) -> torch.Tensor:
        K_star = self.B_to_S(K_B, "k")
        V_star = self.B_to_S(V_B, "v")
        K_A = self.S_to_A(K_star, "k")
        V_A = self.S_to_A(V_star, "v")
        return F.mse_loss(K_A, K_A_target) + F.mse_loss(V_A, V_A_target)

    def forward(self, K_B, V_B, K_A_target, V_A_target) -> torch.Tensor:
        return self.loss(K_B, V_B, K_A_target, V_A_target)

In [11]:
# code1_train_B2A.py
# 목표: gpt2-medium(B) -> gpt2(A) 단방향 translator 학습
# - WikiText 데이터로 2000 step
# - 매 100 step 마다 validation reconstruction loss 표시
# - argparse 금지
# - transformers==4.35.2

import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


# -------------------
# 설정
# -------------------
MODEL_A_NAME = "gpt2"         # target
MODEL_B_NAME = "gpt2-medium"  # source

SEED = 42
CONTEXT_LEN = 64
BATCH_SIZE = 2

TRAIN_STEPS = 1000
VAL_EVERY = 100
VAL_BATCHES = 10

LR = 1e-4
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0

# shared space dims (must be divisible by both 12 and 24)
Q_DIM = 1536
D_MODEL = 256
N_HEADS = 8
DROPOUT = 0.0

SAVE_PATH = "translator_B2A_ckpt.pt"


def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_lm_blocks(tokenizer, texts, block_size: int):
    joined = "\n\n".join([t for t in texts if t is not None])
    ids = tokenizer(joined, return_tensors=None)["input_ids"]
    n_blocks = len(ids) // block_size
    ids = ids[: n_blocks * block_size]
    blocks = [ids[i * block_size : (i + 1) * block_size] for i in range(n_blocks)]
    return blocks


def collate_fn(batch):
    input_ids = torch.tensor(batch, dtype=torch.long)
    attention_mask = torch.ones_like(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask}


@torch.no_grad()
def get_kv(model, input_ids, attention_mask):
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True, return_dict=True)
    K, V = pack_past_key_values(out.past_key_values)
    return K, V


@torch.no_grad()
def eval_val_loss(translator, modelA, modelB, val_loader, device):
    translator.eval()
    total = 0.0
    n = 0
    for i, batch in enumerate(val_loader):
        if i >= VAL_BATCHES:
            break
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # source(B) + target(A)
        K_B, V_B = get_kv(modelB, input_ids, attention_mask)
        K_A, V_A = get_kv(modelA, input_ids, attention_mask)

        loss = translator(K_B, V_B, K_A, V_A)
        total += float(loss.item())
        n += 1

    translator.train()
    return total / max(1, n)


def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_A_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    ds = load_dataset("wikitext", "wikitext-2-raw-v1")
    train_blocks = make_lm_blocks(tokenizer, ds["train"]["text"], CONTEXT_LEN)
    val_blocks = make_lm_blocks(tokenizer, ds["validation"]["text"], CONTEXT_LEN)

    train_loader = DataLoader(train_blocks, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
    val_loader = DataLoader(val_blocks, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, drop_last=False)

    # frozen LMs
    modelA = GPT2LMHeadModel.from_pretrained(MODEL_A_NAME).to(device).eval()
    modelB = GPT2LMHeadModel.from_pretrained(MODEL_B_NAME).to(device).eval()
    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id
    for p in modelA.parameters():
        p.requires_grad_(False)
    for p in modelB.parameters():
        p.requires_grad_(False)

    a_layers, a_hidden = modelA.config.n_layer, modelA.config.n_embd
    b_layers, b_hidden = modelB.config.n_layer, modelB.config.n_embd

    translator = OneWayKVTranslator_B2A(
        b_layers=b_layers, b_hidden=b_hidden,
        a_layers=a_layers, a_hidden=a_hidden,
        q_dim=Q_DIM, d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT
    ).to(device).train()

    opt = torch.optim.AdamW(translator.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    train_iter = iter(train_loader)

    for step in range(1, TRAIN_STEPS + 1):
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.no_grad():
            K_B, V_B = get_kv(modelB, input_ids, attention_mask)
            K_A, V_A = get_kv(modelA, input_ids, attention_mask)

        opt.zero_grad(set_to_none=True)
        loss = translator(K_B, V_B, K_A, V_A)
        loss.backward()
        nn.utils.clip_grad_norm_(translator.parameters(), GRAD_CLIP_NORM)
        opt.step()

        if step % 10 == 0:
            print(f"[train] step={step:4d} loss={loss.item():.6f}")

        if step % VAL_EVERY == 0:
            val_loss = eval_val_loss(translator, modelA, modelB, val_loader, device)
            print(f"[valid] step={step:4d} recon_loss={val_loss:.6f}")

    ckpt = {
        "translator_state": translator.state_dict(),
        "config": {
            "MODEL_A_NAME": MODEL_A_NAME,
            "MODEL_B_NAME": MODEL_B_NAME,
            "Q_DIM": Q_DIM,
            "D_MODEL": D_MODEL,
            "N_HEADS": N_HEADS,
            "DROPOUT": DROPOUT,
            "a_layers": a_layers,
            "a_hidden": a_hidden,
            "b_layers": b_layers,
            "b_hidden": b_hidden,
        },
    }
    torch.save(ckpt, SAVE_PATH)
    print("saved:", SAVE_PATH)


if __name__ == "__main__":
    main()

device: cuda


Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors


[train] step=  10 loss=4.078006
[train] step=  20 loss=3.593749
[train] step=  30 loss=3.465290
[train] step=  40 loss=3.475547
[train] step=  50 loss=3.392505
[train] step=  60 loss=3.441114
[train] step=  70 loss=3.415877
[train] step=  80 loss=3.420438
[train] step=  90 loss=3.340909
[train] step= 100 loss=3.305861
[valid] step= 100 recon_loss=3.281789
[train] step= 110 loss=3.271916
[train] step= 120 loss=3.419480
[train] step= 130 loss=3.148234
[train] step= 140 loss=3.087670
[train] step= 150 loss=3.246404
[train] step= 160 loss=3.181283
[train] step= 170 loss=3.203373
[train] step= 180 loss=3.256010
[train] step= 190 loss=3.149057
[train] step= 200 loss=3.158133
[valid] step= 200 recon_loss=3.203409
[train] step= 210 loss=3.252357
[train] step= 220 loss=3.134978
[train] step= 230 loss=3.216570
[train] step= 240 loss=3.158617
[train] step= 250 loss=3.263228
[train] step= 260 loss=3.218435
[train] step= 270 loss=3.134453
[train] step= 280 loss=3.167818
[train] step= 290 loss=3.092

In [13]:
# code2_infer_B2A.py
# 목표: 학습된 B->A translator 기반 KV 변환 추론 + 디버깅
# - cosine: translated(A) vs original(A)
# - prefix 2개에 대해 생성 비교
# - argparse 금지
# - transformers==4.35.2

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast


CKPT_PATH = "translator_B2A_ckpt.pt"
MAX_NEW_TOKENS = 30

PREFIXES = [
    "Seoul is the capital of",
    "Paris is the capital of",
]


@torch.no_grad()
def get_past_excluding_last_token(model, input_ids: torch.Tensor):
    """
    past만으로 generate 시작:
    - past는 prefix[:-1] 캐시
    - 첫 step 입력은 prefix[-1]
    """
    assert input_ids.size(1) >= 2
    out = model(input_ids=input_ids[:, :-1], use_cache=True, return_dict=True)
    return out.past_key_values


@torch.no_grad()
def greedy_generate_from_past(model, tokenizer, prefix_ids, past_excl_last, max_new_tokens: int):
    model.eval()
    input_ids = prefix_ids[:, -1:]
    generated = prefix_ids.clone()
    past = past_excl_last

    for _ in range(max_new_tokens):
        out = model(input_ids=input_ids, past_key_values=past, use_cache=True, return_dict=True)
        logits = out.logits[:, -1, :]
        next_id = torch.argmax(logits, dim=-1, keepdim=True)
        generated = torch.cat([generated, next_id], dim=1)
        past = out.past_key_values
        input_ids = next_id

    return tokenizer.decode(generated[0], skip_special_tokens=True)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)

    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    cfg = ckpt["config"]

    tokenizer = GPT2TokenizerFast.from_pretrained(cfg["MODEL_A_NAME"])
    tokenizer.pad_token = tokenizer.eos_token

    modelA = GPT2LMHeadModel.from_pretrained(cfg["MODEL_A_NAME"]).to(device).eval()
    modelB = GPT2LMHeadModel.from_pretrained(cfg["MODEL_B_NAME"]).to(device).eval()
    modelA.config.pad_token_id = tokenizer.eos_token_id
    modelB.config.pad_token_id = tokenizer.eos_token_id

    translator = OneWayKVTranslator_B2A(
        b_layers=cfg["b_layers"], b_hidden=cfg["b_hidden"],
        a_layers=cfg["a_layers"], a_hidden=cfg["a_hidden"],
        q_dim=cfg["Q_DIM"], d_model=cfg["D_MODEL"], n_heads=cfg["N_HEADS"], dropout=cfg["DROPOUT"]
    ).to(device).eval()
    translator.load_state_dict(ckpt["translator_state"])

    specA = ModelKVSpec(
        n_layers=modelA.config.n_layer,
        n_heads=modelA.config.n_head,
        head_dim=modelA.config.n_embd // modelA.config.n_head,
        hidden_size=modelA.config.n_embd,
    )

    for prefix in PREFIXES:
        print("\n" + "=" * 80)
        print("PREFIX:", prefix)

        prefix_ids = tokenizer(prefix, return_tensors="pt").input_ids.to(device)
        if prefix_ids.size(1) < 2:
            print("prefix token length too short; skip")
            continue

        # original past (excluding last token)
        pastA = get_past_excluding_last_token(modelA, prefix_ids)
        pastB = get_past_excluding_last_token(modelB, prefix_ids)

        K_A_true, V_A_true = pack_past_key_values(pastA)
        K_B, V_B = pack_past_key_values(pastB)

        # translate B -> A
        with torch.no_grad():
            K_A_pred, V_A_pred = translator.translate(K_B, V_B)

        # cosine debug: predicted(A) vs true(A)
        cos_k = cosine_sim_flat(K_A_pred, K_A_true)
        cos_v = cosine_sim_flat(V_A_pred, V_A_true)
        print(f"[cosine] translated(A) vs original(A): key={cos_k:.6f}, val={cos_v:.6f}")

        # build past for model A from translated KV
        pastA_from_B = unpack_past_key_values(K_A_pred, V_A_pred, specA)

        # generations
        out1 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA, MAX_NEW_TOKENS)
        out2 = greedy_generate_from_past(modelA, tokenizer, prefix_ids, pastA_from_B, MAX_NEW_TOKENS)
        out3 = greedy_generate_from_past(modelB, tokenizer, prefix_ids, pastB, MAX_NEW_TOKENS)

        print("\n[generate] (1) A_original -> model A")
        print(out1)
        print("\n[generate] (2) B_to_A     -> model A")
        print(out2)
        print("\n[generate] (3) B_original -> model B")
        print(out3)


if __name__ == "__main__":
    main()

device: cuda

PREFIX: Seoul is the capital of
[cosine] translated(A) vs original(A): key=0.618845, val=0.490790

[generate] (1) A_original -> model A
Seoul is the capital of South Korea, and is home to the country's largest and most famous tourist attraction.

The city is also home to the country's largest and

[generate] (2) B_to_A     -> model A
Seoul is the capital of the world.


The

The

The
The

The
The

The
The

The
The

[generate] (3) B_original -> model B
Seoul is the capital of South Korea, and is home to the country's largest concentration of foreign-born residents.

The city's population is estimated at around 1.

PREFIX: Paris is the capital of
[cosine] translated(A) vs original(A): key=0.627573, val=0.518963

[generate] (1) A_original -> model A
Paris is the capital of the world's largest oil-rich state, and the world's second-largest oil producer.

The country's oil production is expected to reach

[generate] (2) B_to_A     -> model A
Paris is the capital of the United 